In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from weather.config import *
from weather.utils.load_models import load_all_weather_models
from weather.dataset.dataloader import get_weather_loaders
from weather.dataset.augmentation import generate_manual_aug
from weather.dataset.generation import generate_ai_weather
from weather.utils.attention import show_attention

In [ ]:
data, loaders, labels, class_names = get_weather_loaders(DATA_PATH)

In [ ]:
models = load_all_weather_models(class_names=class_names)

# Создание данных

## Аугментация данных

In [ ]:
generate_manual_aug(MY_DATA_TEST_PATH, AUG_DATA_TEST_PATH)

## Генерация данных

In [ ]:
generate_ai_weather(GEN_DATA_TEST_PATH, class_names)

# Проверка моделей

In [ ]:
data_groups = {
    "Свои данные": MY_DATA_TEST_PATH,
    "Аугментированные данные": AUG_DATA_TEST_PATH,
    "Сгенерированные данные": GEN_DATA_TEST_PATH,
}

In [ ]:
from weather.utils.test_model import test_model

for group_name, path in data_groups.items():
    print(group_name)
    for name in models:
        if name != "ViT":
            test_model(
                model=models[name],
                model_name=name,
                data_path=path,
                class_names=class_names,
                device=DEVICE
            )
        else:
            test_model(
                model=models[name]["model"],
                model_name=name,
                data_path=path,
                class_names=class_names,
                device=DEVICE
            )
    print("\n\n\n")

# Внимание

In [ ]:
test_img, test_label = data['test'][0]

for name in models:
    print(f"\nАнализируем модель: {name}")

    m = models[name]
    if isinstance(m, dict) and "model" in m:
        m = m["model"]

    size = 224 if "vit" in name.lower() else 128
    current_img = torch.nn.functional.interpolate(test_img.unsqueeze(0), size=(size, size)).to(DEVICE).squeeze(0)

    try:
        show_attention(m, name, current_img, test_label, class_names, DEVICE)
    except Exception as e:
        print(f"Ошибка при визуализации {name}: {e}")